# install dependencies

In [19]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.model_selection import KFold
import statsmodels.api as sm
import pyreadstat

# Phase1 Data Preparation

In [20]:
def calculate_cir_series(
    outcome,
    file_name,
    data_path='../../result/occ/analysis',
    horizon_start=1,
    horizon_end=36,
    series_name=None,
    standardize=True
):

    file_path = f"{data_path.rstrip('/\\\\')}/{file_name}"
    df = pd.read_csv(file_path)

    if series_name is None:
        series_name = f"CIR_{horizon_end}"

    df_outcome = df[df['outcome'] == outcome].copy()
    df_outcome['horizon'] = pd.to_numeric(df_outcome['horizon'])
    df_outcome = df_outcome.sort_values('horizon')

    group_cols = [col for col in df_outcome.columns if col.startswith('group')]

    cir_dict = {}

    window = df_outcome[df_outcome['horizon'].between(horizon_start, horizon_end)]

    # raw CIR
    for col in group_cols:
        cir_dict[col] = window[col].sum()

    cir_series = pd.Series(cir_dict, name=series_name)

    # ✔ cross-group standardization
    if standardize:
        cir_series = (cir_series - cir_series.mean()) / cir_series.std()

    return cir_series

In [21]:
def build_y_series_from_mapping(
    cir_series,
    file_name=None,
    data_path=None,
    series_name=None
):
    """
    直接构造 group-level y_series
    不再做 SOC-level mapping
    """

    if series_name is None:
        series_name = cir_series.name if cir_series.name is not None else 'value'

    # group → value
    group_to_value = {}

    for idx, val in cir_series.items():
        match = re.search(r'group(\d+)', str(idx))
        if match:
            group_to_value[int(match.group(1))] = val

    y_series = pd.Series(group_to_value, name=series_name)

    y_series.index.name = "group"

    return y_series

In [22]:
def load_and_prepare_onet_data_extended(
    y_series,
    file_names,
    mapping_path,
    onet_data_path='../../data/ONET',
    mapping_sheet='Sheet1',
    scale_id='IM',
    usecols=[0, 1, 4, 5, 7]
):

    import pandas as pd
    import numpy as np
    from sklearn.preprocessing import StandardScaler

    # ── 1. 读取 mapping（新增 Group + Weight） ─────────────────────
    df_map = pd.read_excel(
        mapping_path,
        sheet_name=mapping_sheet,
        usecols='A,B,E,F,H',   # ⭐新增 F + H
        header=0
    )

    df_map.columns = ['occ1990', 'occ1990dd', 'SOC-2018', 'group', 'weight']

    df_map['occ1990'] = pd.to_numeric(df_map['occ1990'], errors='coerce').astype('Int64')

    df_map['SOC-2018'] = df_map['SOC-2018'].astype(str).str.strip()

    df_map['group'] = pd.to_numeric(df_map['group'], errors='coerce').astype('Int64')

    df_map['weight'] = pd.to_numeric(df_map['weight'], errors='coerce')

    df_map = df_map.dropna(subset=['SOC-2018', 'group', 'weight'])

    valid_soc = set(df_map['SOC-2018'].unique())

    print(f"mapping 中有效 SOC-2018 数量: {len(valid_soc)}")

    # ── 2. 读取 O*NET（不变） ─────────────────────────────────────
    dfs = []

    for prefix, fname in file_names.items():
        fpath = f"{onet_data_path.rstrip('/\\\\')}/{fname}"

        df = pd.read_excel(fpath, usecols=usecols, header=0)

        df.columns = [
            'SOC_Code',
            'Sub_Code',
            'Element_Name',
            'Scale_ID',
            'Data_Value'
        ]

        df['SOC_Code'] = df['SOC_Code'].astype(str).str.strip()
        df['Element_Name'] = df['Element_Name'].astype(str).str.strip()
        df['Scale_ID'] = df['Scale_ID'].astype(str).str.strip().str.upper()
        df['Sub_Code'] = df['Sub_Code'].astype(str).str.strip().str.zfill(2)

        means = df.groupby(['SOC_Code', 'Element_Name'])['Data_Value'].mean().reset_index()
        means.rename(columns={'Data_Value': 'Mean_Val'}, inplace=True)

        df = df.merge(means, on=['SOC_Code', 'Element_Name'], how='left')

        df.loc[df['Sub_Code'] == '00', 'Data_Value'] = df.loc[df['Sub_Code'] == '00', 'Mean_Val']

        df = df[df['Sub_Code'] == '00'].copy()
        df = df[df['Scale_ID'] == scale_id].copy()
        df = df.dropna(subset=['Data_Value'])

        df.drop(columns=['Mean_Val', 'Sub_Code', 'Scale_ID'], inplace=True)

        df = df[df['SOC_Code'].isin(valid_soc)].copy()

        df['Element_Name'] = prefix + '_' + df['Element_Name']

        dfs.append(df)

    # ── 3. merge O*NET ─────────────────────────────────────────────
    df_all = pd.concat(dfs, ignore_index=True)

    df_merged = df_all.merge(
        df_map[['SOC-2018', 'group', 'weight']],
        left_on='SOC_Code',
        right_on='SOC-2018',
        how='inner'
    )

    # ── ⭐ 4. weighted mean by group ────────────────────────────────
    df_weighted = (
        df_merged
        .groupby(['group', 'Element_Name'])
        .apply(lambda x: np.average(x['Data_Value'], weights=x['weight']))
        .reset_index(name='Data_Value')
    )

    # ── 5. pivot ────────────────────────────────────────────────────
    df_wide = df_weighted.pivot_table(
        index='group',
        columns='Element_Name',
        values='Data_Value'
    ).astype(float)

    df_wide = df_wide.fillna(df_wide.median())

    print(
        f"O*NET weighted group-level: "
        f"{df_wide.shape[0]} groups, {df_wide.shape[1]} features"
    )

    # ── 6. standardization ─────────────────────────────────────────
    scaler = StandardScaler()

    X_df = pd.DataFrame(
        scaler.fit_transform(df_wide),
        columns=df_wide.columns,
        index=df_wide.index
    )

    # ── 7. align y (IMPORTANT: y must also be group-level now) ─────
    aligned_idx = X_df.index.intersection(y_series.index)

    X = X_df.loc[aligned_idx].values
    y_aligned = y_series.loc[aligned_idx].values

    print(f"X shape: {X.shape} | Aligned samples: {len(aligned_idx)}")

    return X_df, aligned_idx, X, y_aligned

# Phase2: correlation

In [23]:
from scipy.stats import pearsonr, spearmanr

def run_bivariate_correlations(X_df, y_aligned, top_n=10):
    results = []
    for col in X_df.columns:
        x = X_df[col].values
        r_p, p_p = pearsonr(x, y_aligned)
        r_s, p_s = spearmanr(x, y_aligned)
        results.append({
            "feature": col,
            "pearson_r": r_p,
            "pearson_p": p_p,
            "spearman_r": r_s,
            "spearman_p": p_s,
            "abs_pearson": abs(r_p)
        })
    
    df = pd.DataFrame(results).sort_values("abs_pearson", ascending=False)
    
    print(f"\nTop {top_n} by |Pearson r|:")
    print(df.head(top_n)[["feature","pearson_r","pearson_p","spearman_r","spearman_p"]].round(3).to_string())
    
    return df

# Main

In [24]:

def main():

    irf_file = "merged_occ_irf_trajectories.csv"
    mapping_file = "mapping_done.xlsx"

    file_sets = {
        "Abilities": {"Abilities": "Abilities.xlsx"},
        "Knowledge": {"Knowledge": "Knowledge.xlsx"},
        "Skills": {"Skills": "Skills.xlsx"}
    }

    outcomes = [
        'unemployment', 'employment', 'income', 'hourly_rate',
        'hours', 'income_share', 'inequality', 'median'
    ]

    all_corr_rows = []      # full correlation table (all features, all outcomes, all tables)
    top10_rows = []         # top 10 per outcome × table

    for outcome in outcomes:

        print("\n" + "=" * 100)
        print(f"OUTCOME: {outcome}")
        print("=" * 100)

        cir_series = calculate_cir_series(
            outcome=outcome,
            file_name=irf_file,
            horizon_start=1,
            horizon_end=36,
            series_name=f"{outcome}_CIR36"
        )

        y_series = build_y_series_from_mapping(
            cir_series=cir_series,
            series_name=outcome
        )

        for table_name, file_dict in file_sets.items():

            print("\n" + "-" * 60)
            print(f"{outcome} | {table_name}")
            print("-" * 60)

            X_df, aligned_idx, X, y_aligned = load_and_prepare_onet_data_extended(
                y_series=y_series,
                file_names=file_dict,
                mapping_path=f"../../result/mapping/{mapping_file}"
            )

            corr_df = run_bivariate_correlations(X_df, y_aligned, top_n=10)

            # tag with outcome + table
            corr_df["outcome"] = outcome
            corr_df["table"] = table_name

            all_corr_rows.append(corr_df)

            # top 10 by |spearman_r|
            top10 = corr_df.head(10).copy()
            top10_rows.append(top10)

            print(f"\nTop 10 by |Spearman r| (n=9):")
            print(
                top10[["feature", "spearman_r", "spearman_p", "pearson_r", "pearson_p"]]
                .round(3)
                .to_string(index=False)
            )

    # --------------------------------------------------
    # save
    # --------------------------------------------------
    full_corr_df = pd.concat(all_corr_rows, ignore_index=True)
    top10_df = pd.concat(top10_rows, ignore_index=True)

    # reorder columns for clarity
    col_order = ["outcome", "table", "feature", "spearman_r", "spearman_p", "pearson_r", "pearson_p", "abs_spearman"]
    full_corr_df = full_corr_df[col_order]
    top10_df = top10_df[col_order]

    full_corr_df.to_excel("bivariate_correlations_full.xlsx", index=False)
    top10_df.to_excel("bivariate_top10.xlsx", index=False)

    print("\n" + "=" * 100)
    print("DONE. Saved:")
    print("  1. bivariate_correlations_full.xlsx  — all features ranked per outcome × table")
    print("  2. bivariate_top10.xlsx              — top 10 per outcome × table")


if __name__ == "__main__":
    main()


OUTCOME: unemployment

------------------------------------------------------------
unemployment | Abilities
------------------------------------------------------------
mapping 中有效 SOC-2018 数量: 663
O*NET weighted group-level: 9 groups, 52 features
X shape: (9, 52) | Aligned samples: 9

Top 10 by |Pearson r|:
                             feature  pearson_r  pearson_p  spearman_r  spearman_p
24             Abilities_Near Vision      0.902      0.001       0.917       0.001
19    Abilities_Information Ordering      0.837      0.005       0.883       0.002
2     Abilities_Category Flexibility      0.835      0.005       0.900       0.001
36     Abilities_Selective Attention      0.822      0.007       0.633       0.067
12  Abilities_Flexibility of Closure      0.778      0.013       0.700       0.036
18     Abilities_Inductive Reasoning      0.739      0.023       0.850       0.004
4      Abilities_Deductive Reasoning      0.734      0.024       0.883       0.002
30        Abilities_Perc

KeyError: "['abs_spearman'] not in index"